# Chapter 2: Data Collection Techniques

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aeleraqi/Python-for-Journalists/blob/main/notebooks/Chapter_02_Notebook.ipynb)

Companion notebook to *The Code Behind the Story: Data Skills with Python for Journalists*, second edition, by Amr Eleraqi. It holds every code cell of the chapter, in the book's order; the book explains each one. Run the cells from top to bottom.

## 2.1 Why data collection matters

## 2.2 Types of data and structures of data

### 2.2.1 Types of data

In [ ]:
import pandas as pd

worker = pd.DataFrame({
    "Name": ["Ali"],
    "Age": [28],
    "Salary": [25000.50],
    "Employed": ["Employed"]
})
worker.dtypes

In [ ]:
worker["Employed"] == "Employed"

### 2.2.2 Structured, unstructured and semi-structured data

In [ ]:
import json

patients_json = """
[
  {"Blood type": "A", "Gender": "Male", "Floor": 1, "Disease": "Influenza"},
  {"Blood type": "B", "Gender": "Female", "Floor": 3, "Disease": "Cancer"}
]
"""
patients = json.loads(patients_json)
pd.DataFrame(patients)

## 2.3 Common file formats

In [ ]:
with open("Egyptian Doctors Data.csv", encoding="utf-8-sig") as file:
    header = file.readline()
    first_row = file.readline()
print(len(header.split(",")), "column names")
first_row.split(",")

## 2.4 Importing data files

In [ ]:
from google.colab import files

uploaded = files.upload()

### 2.4.1 Importing Excel files

In [ ]:
import pandas as pd

doctors_xlsx = pd.read_excel("Egyptian Doctors Data.xlsx")
doctors_xlsx.shape

In [ ]:
doctors_xlsx[["specialization", "fees", "avg_rate", "clinic_location"]].head()

In [ ]:
with pd.ExcelFile("Egyptian Doctors Data.xlsx") as workbook:
    print(workbook.sheet_names)

In [ ]:
pages = pd.read_excel("Egyptian Doctors Data.xlsx",
                      usecols=["specialization", "fees", "pages"],
                      na_values=["..."])
pages.iloc[3:8]

In [ ]:
# This cell stops with an error on purpose: read the message.
pd.read_excel("Egyptian Doctors Data.xlsx", chunksize=500)

### 2.4.2 Importing CSV files

In [ ]:
doctors_csv = pd.read_csv("Egyptian Doctors Data.csv")
doctors_csv.shape

In [ ]:
without_last_rows = pd.read_csv("Egyptian Doctors Data.csv",
                                skipfooter=2, engine="python")
without_last_rows.shape

In [ ]:
dentists = 0
with pd.read_csv("Egyptian Doctors Data.csv", chunksize=500) as reader:
    for chunk in reader:
        print(chunk.shape)
        dentists += (chunk["specialization"] == "Dentist").sum()
print("Dentists:", dentists)

In [ ]:
wrong = pd.read_csv("Egyptian Doctors Data.csv", encoding="latin1")
wrong.columns[:3]

### 2.4.3 Importing text files

In [ ]:
with open("Donald Trump.txt", encoding="utf-8") as file:
    speech = file.read()

print(len(speech))
print(speech[:64])

In [ ]:
with open("Donald Trump.txt", encoding="utf-8") as file:
    lines = file.readlines()

print(len(lines))
lines[:4]

In [ ]:
paragraphs = []
for line in lines:
    if line.strip():
        paragraphs.append(line.strip())
len(paragraphs)

In [ ]:
from itertools import islice

with open("Donald Trump.txt", encoding="utf-8") as file:
    for line in islice(file, 8, 12):
        print(line.strip()[:70])

In [ ]:
turns = []
for line in lines:
    line = line.strip()
    if line.endswith("):"):
        speaker, stamp = line[:-2].split(" (")
        turns.append({"speaker": speaker, "time": stamp})

turns = pd.DataFrame(turns)
turns["speaker"].value_counts()

In [ ]:
print(speech.count("America"))
print(speech.count("American"))

### 2.4.4 Reading JSON

In [ ]:
record_json = (
    '{"author": {"goodreads_link": "/author/show/9810.Albert_Einstein",'
    ' "name": "Albert Einstein", "slug": "Albert-Einstein"},'
    ' "tags": ["change", "deep-thoughts", "thinking", "world"],'
    ' "text": "“The world as we have created it is a process of our'
    ' thinking. It cannot be changed without changing our thinking.”"}'
)
record = json.loads(record_json)
print(record["author"]["name"])
print(record["tags"][:2])

In [ ]:
flat = pd.json_normalize([record])
list(flat.columns)

### 2.4.5 Saving your work

In [ ]:
sample = doctors_csv[["specialization", "fees", "avg_rate"]].head(3)
sample.to_csv("doctors_sample.csv")
pd.read_csv("doctors_sample.csv")

In [ ]:
sample.to_csv("doctors_sample.csv", index=False)
pd.read_csv("doctors_sample.csv")

In [ ]:
sample.to_excel("doctors_sample.xlsx", sheet_name="Sample", index=False)

In [ ]:
sample.to_json("doctors_sample.json", orient="records", indent=2)
with open("doctors_sample.json", encoding="utf-8") as file:
    print(file.read()[:150])

## 2.5 Colab and outside sources

### 2.5.1 Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
doctors = pd.read_excel("/content/drive/MyDrive/Egyptian Doctors Data.xlsx")

In [ ]:
doctors.to_csv("/content/drive/MyDrive/doctors_clean.csv", index=False)

In [ ]:
drive.flush_and_unmount()

### 2.5.2 Kaggle

In [ ]:
handle = "valakhorasani/mobile-device-usage-and-user-behavior-dataset"

In [ ]:
import kagglehub
import shutil

path = kagglehub.dataset_download(handle)
print(path)
shutil.copy(path + "/user_behavior_dataset.csv", "/content")

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

In [ ]:
!kaggle datasets download {handle} --unzip

In [ ]:
users = pd.read_csv("user_behavior_dataset.csv")
users.shape

In [ ]:
users[["Device Model", "Operating System", "App Usage Time (min/day)",
       "Age", "Gender"]].head()

### 2.5.3 GitHub

In [ ]:
url = ("https://raw.githubusercontent.com/aeleraqi/Data/main/"
       "Egyptian%20Doctors%20Data.csv")
doctors_github = pd.read_csv(url)
doctors_github.shape

In [ ]:
doctors_github.equals(doctors_csv)

In [ ]:
guns_url = ("https://raw.githubusercontent.com/fivethirtyeight/guns-data/"
            "master/interactive_data.csv")
guns = pd.read_csv(guns_url)
guns.columns[:3]

In [ ]:
guns = pd.read_csv(guns_url, index_col=0)
guns.head(3)

In [ ]:
all_people = ((guns["Gender"] == "None selected")
              & (guns["Age"] == "None selected")
              & (guns["Race"] == "None selected"))
guns[all_people]

In [ ]:
guns["Age"].unique()

## 2.6 Combining DataFrames

### 2.6.1 Checking before you combine

In [ ]:
list(doctors_csv.columns) == list(doctors_xlsx.columns)

In [ ]:
doctors_csv.dtypes.equals(doctors_xlsx.dtypes)

In [ ]:
doctors_both = pd.concat([doctors_csv, doctors_xlsx], ignore_index=True)
doctors_both.shape

In [ ]:
print("Repeated rows:", doctors_both.duplicated().sum())
print("Rows left:", len(doctors_both.drop_duplicates()))

In [ ]:
print(doctors_csv["pages"].head(3).tolist())
print(doctors_xlsx["pages"].head(3).tolist())

In [ ]:
print(doctors_csv.loc[107, "clinic_location"])
print(doctors_xlsx.loc[107, "clinic_location"])

### 2.6.2 Combining many files

In [ ]:
import os

os.makedirs("by_specialty", exist_ok=True)
for specialty in ["Dentist", "Gynecologist", "Cardiologist"]:
    rows = doctors_csv[doctors_csv["specialization"] == specialty]
    rows.to_csv(f"by_specialty/{specialty}.csv", index=False)

In [ ]:
import glob

paths = sorted(glob.glob("by_specialty/*.csv"))
paths

In [ ]:
pieces = []
for path in paths:
    piece = pd.read_csv(path)
    piece["source_file"] = path
    pieces.append(piece)

three = pd.concat(pieces, ignore_index=True)
three.shape

### 2.6.3 Side by side: merge

## 2.7 Web scraping

### 2.7.1 How scraping works

### 2.7.2 Before you scrape: rules and manners

In [ ]:
from urllib.robotparser import RobotFileParser

rules = """
User-agent: *
Disallow: /search
Disallow: /admin/
Crawl-delay: 5
"""
robots = RobotFileParser()
robots.parse(rules.splitlines())

print(robots.can_fetch("*", "https://example.com/news/budget-2026"))
print(robots.can_fetch("*", "https://example.com/search?q=budget"))
print(robots.crawl_delay("*"))

### 2.7.3 Tools for scraping

### 2.7.4 Fetching a page

In [ ]:
import requests

headers = {"User-Agent": "Newsroom training script (you@example.org)"}

url = "https://quotes.toscrape.com/page/1/"
response = requests.get(url, headers=headers, timeout=30)
response.status_code

In [ ]:
print(response.headers["Content-Type"])
print(response.text[:160])

In [ ]:
missing = requests.get("https://quotes.toscrape.com/no-such-page/",
                       headers=headers, timeout=30)
missing.status_code

In [ ]:
# This cell stops with an error on purpose: read the message.
missing.raise_for_status()

### 2.7.5 Reading the HTML

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")
quotes_on_page = soup.find_all("div", class_="quote")
len(quotes_on_page)

In [ ]:
print(quotes_on_page[9].prettify())

In [ ]:
first = quotes_on_page[0]

text = first.find("span", class_="text").get_text(strip=True)
author = first.find("small", class_="author").get_text(strip=True)
tags = [a.get_text() for a in first.find_all("a", class_="tag")]

print(text[:60])
print(author)
print(tags)

In [ ]:
print(len(soup.select("a.tag")))
print(len(soup.select("div.quote a.tag")))

In [ ]:
def parse_quotes(soup):
    """Return a list of dictionaries, one for each quote on the page."""
    rows = []
    for quote in soup.find_all("div", class_="quote"):
        text = quote.find("span", class_="text").get_text(strip=True)
        author = quote.find("small", class_="author").get_text(strip=True)
        tags = [a.get_text() for a in quote.find_all("a", class_="tag")]
        rows.append({"text": text.strip("“”"),
                     "author": author,
                     "tags": ",".join(tags)})
    return rows

page_one = pd.DataFrame(parse_quotes(soup))
page_one[["author", "tags"]].head()

### 2.7.6 Many pages

In [ ]:
page_11 = requests.get("https://quotes.toscrape.com/page/11/",
                       headers=headers, timeout=30)
print(page_11.status_code)
eleven = BeautifulSoup(page_11.text, "html.parser")
print(len(eleven.find_all("div", class_="quote")))

In [ ]:
import time
from urllib.parse import urljoin

url = "https://quotes.toscrape.com/page/1/"
rows = []
while url:
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    rows.extend(parse_quotes(soup))
    next_link = soup.select_one("li.next a")
    if next_link:
        url = urljoin(url, next_link["href"])
        time.sleep(1)
    else:
        url = None

quotes = pd.DataFrame(rows)
quotes.shape

In [ ]:
print(quotes["author"].nunique(), "authors")
quotes["author"].value_counts().head()

In [ ]:
quotes["tags"].str.split(",").explode().value_counts().head()

In [ ]:
quotes.to_csv("quotes.csv", index=False)
pd.read_csv("quotes.csv").shape

### 2.7.7 Author pages and a merge

In [ ]:
author_pages = ["Albert-Einstein", "J-K-Rowling", "Jane-Austen",
                "Marilyn-Monroe", "Mark-Twain"]

author_rows = []
for page in author_pages:
    url = f"https://quotes.toscrape.com/author/{page}/"
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    name = soup.find("h3", class_="author-title").get_text(strip=True)
    born = soup.find("span", class_="author-born-date").get_text()
    place = soup.find("span", class_="author-born-location").get_text()
    author_rows.append({"author": name, "born": born, "born_in": place})
    time.sleep(1)

authors = pd.DataFrame(author_rows)
authors

In [ ]:
authors["born_in"] = authors["born_in"].str.removeprefix("in ")

In [ ]:
quotes_with_authors = quotes.merge(authors, on="author", how="left")
print(quotes_with_authors.shape)
print(quotes_with_authors["born"].notna().sum(), "quotes with a birth date")

In [ ]:
quotes_with_authors[["author", "born", "born_in"]].head(3)

## 2.8 RSS feeds

### 2.8.1 Reading a feed

In [ ]:
feed_url = "https://www.federalreserve.gov/feeds/press_all.xml"
response = requests.get(feed_url, headers=headers, timeout=30)
print(response.status_code, response.headers["Content-Type"])

In [ ]:
feed = BeautifulSoup(response.content, "xml")
print(feed.channel.title.get_text())
items = feed.find_all("item")
len(items)

In [ ]:
first_item = items[0]
for name in ["title", "category", "pubDate", "link"]:
    print(name, ":", first_item.find(name).get_text()[:60])

In [ ]:
release_rows = []
for item in items:
    release_rows.append({
        "title": item.find("title").get_text(),
        "category": item.find("category").get_text(),
        "published": item.find("pubDate").get_text(),
        "description": item.find("description").get_text(),
        "link": item.find("link").get_text(),
    })

releases = pd.DataFrame(release_rows)
releases.shape

In [ ]:
releases["published"] = pd.to_datetime(releases["published"], utc=True)
print(releases["published"].min())
print(releases["published"].max())

In [ ]:
releases["category"].value_counts()

In [ ]:
different = releases[releases["title"] != releases["description"]]
for value in different["description"]:
    print(value)

In [ ]:
import html

releases["description"] = releases["description"].map(html.unescape)
print((releases["title"] != releases["description"]).sum())

### 2.8.2 Google News search feeds

In [ ]:
from urllib.parse import quote_plus

search = '"data journalism" when:7d'
news_url = ("https://news.google.com/rss/search?q=" + quote_plus(search)
            + "&hl=en-US&gl=US&ceid=US:en")
response = requests.get(news_url, headers=headers, timeout=30)
news = BeautifulSoup(response.content, "xml")

news_rows = []
for item in news.find_all("item"):
    news_rows.append({"title": item.find("title").get_text(),
                      "source": item.find("source").get_text(),
                      "published": item.find("pubDate").get_text()})
news_items = pd.DataFrame(news_rows)
news_items.head()

## 2.9 Extracting the text of articles

In [ ]:
!pip install trafilatura

In [ ]:
import trafilatura

url = "https://quotes.toscrape.com/author/Jane-Austen/"
response = requests.get(url, headers=headers, timeout=30)
main_text = trafilatura.extract(response.text)

for line in main_text.splitlines()[:3]:
    print(line)

In [ ]:
page_text = BeautifulSoup(response.text, "html.parser").get_text()
for word in ["Login", "Zyte", "Jane Austen"]:
    print(word, word in page_text, word in main_text)

In [ ]:
meta = trafilatura.extract_metadata(response.text)
print(meta.title)
print(meta.author)
print(meta.date)

## 2.10 Hands-on exercises

### Exercise 1: Data from GitHub

In [ ]:
import pandas as pd

guns_url = ("https://raw.githubusercontent.com/fivethirtyeight/guns-data/"
            "master/interactive_data.csv")
guns = pd.read_csv(guns_url, index_col=0)

homicides = guns[(guns["Intent"] == "Homicide")
                 & (guns["Gender"] == "None selected")
                 & (guns["Age"] == "None selected")]
homicides[["Race", "Deaths", "Population", "Rate"]]

### Exercise 2: One table, three formats

In [ ]:
import os

quotes = pd.read_csv("quotes.csv")
quotes.to_csv("quotes_copy.csv", index=False)
quotes.to_excel("quotes_copy.xlsx", index=False)
quotes.to_json("quotes_copy.json", orient="records")

print(os.path.getsize("quotes_copy.csv"))
print(os.path.getsize("quotes_copy.xlsx"))
print(os.path.getsize("quotes_copy.json"))

print(pd.read_csv("quotes_copy.csv").shape)
print(pd.read_excel("quotes_copy.xlsx").shape)
print(pd.read_json("quotes_copy.json").shape)

### Exercise 3: Reading a feed

In [ ]:
import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Newsroom training script (you@example.org)"}
response = requests.get("https://www.federalreserve.gov/feeds/press_all.xml",
                        headers=headers, timeout=30)
feed = BeautifulSoup(response.content, "xml")

rows = []
for item in feed.find_all("item"):
    rows.append({"category": item.find("category").get_text(),
                 "published": item.find("pubDate").get_text()})
fed = pd.DataFrame(rows)
fed["published"] = pd.to_datetime(fed["published"], utc=True)

september = fed[(fed["published"] >= "2026-09-01")
                & (fed["published"] < "2026-10-01")]
print(len(september), "releases in September")
september["category"].value_counts()

## 2.11 Project: a tracker for a regulator's press releases

### Step 1: A function that reads the feed

In [ ]:
import html
import requests
import pandas as pd
from bs4 import BeautifulSoup

FEED_URL = "https://www.federalreserve.gov/feeds/press_all.xml"
HEADERS = {"User-Agent": "Newsroom training script (you@example.org)"}

def fetch_releases(feed_url=FEED_URL):
    """Download a feed and return its items as a DataFrame."""
    response = requests.get(feed_url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    feed = BeautifulSoup(response.content, "xml")
    rows = []
    for item in feed.find_all("item"):
        rows.append({
            "title": html.unescape(item.find("title").get_text()),
            "category": item.find("category").get_text(),
            "published": item.find("pubDate").get_text(),
            "link": item.find("link").get_text(),
        })
    releases = pd.DataFrame(rows)
    releases["published"] = pd.to_datetime(releases["published"], utc=True)
    return releases

latest = fetch_releases()
latest.shape

### Step 2: An archive that grows

In [ ]:
import os

def update_archive(new, path="fed_archive.csv"):
    """Add the rows of `new` that are not in the archive yet."""
    if os.path.exists(path):
        archive = pd.read_csv(path, parse_dates=["published"])
        before = len(archive)
        archive = pd.concat([archive, new], ignore_index=True)
        archive = archive.drop_duplicates(subset="link", keep="first")
    else:
        before = 0
        archive = new
    archive.to_csv(path, index=False)
    print(f"{len(archive) - before} new, {len(archive)} in the archive")
    return archive

In [ ]:
archive = update_archive(latest)

In [ ]:
archive = update_archive(fetch_releases())

### Step 3: Ask the archive questions

In [ ]:
import matplotlib.pyplot as plt

counts = archive["category"].value_counts(ascending=True)
counts.plot(kind="barh")
plt.title("Federal Reserve Board press releases by category")
plt.xlabel("Number of releases")
plt.ylabel("Category")
plt.show()

In [ ]:
monetary = archive[archive["category"] == "Monetary Policy"]
monetary = monetary.sort_values("published", ascending=False)
for date, title in zip(monetary["published"], monetary["title"]):
    print(date.date(), title[:64])

### Step 4: Make it yours

## 2.12 Habits worth keeping

## Solutions to the "Your turn" exercises

### Section 2.2: Types of data and structures of data

In [ ]:
# 1
# city: text; date: text, better converted to a date;
# participants: integer; arrests_reported: text, convertible to a Boolean;
# crowd_density: float
import json
import pandas as pd

In [ ]:
# 2
workers = pd.DataFrame({
    "Name": ["Ali", "Mona"],
    "Age": [28, 35],
    "Salary": [25000.50, 0.0],
    "Employed": ["Employed", "Unemployed"]
})
workers["is_employed"] = workers["Employed"] == "Employed"
print(workers)

In [ ]:
# 3
people = json.loads("""
[{"name": "Amr", "beat": "Data", "years": 20},
 {"name": "Sara", "beat": "Economy", "years": 6}]
""")
pd.DataFrame(people)

### Section 2.4: Importing data files

In [ ]:
# 1
doctors = pd.read_csv("Egyptian Doctors Data.csv", na_values=["..."])
print(doctors["pages"].isna().sum())

In [ ]:
# 2
count = 0
with open("Donald Trump.txt", encoding="utf-8") as file:
    for line in file:
        if "country" in line.lower():
            count += 1
print(count)

In [ ]:
# 3
turns.to_csv("turns.csv", index=False)
pd.read_csv("turns.csv").shape

### Section 2.5: Colab and outside sources

In [ ]:
# 1
import requests

url = ("https://raw.githubusercontent.com/aeleraqi/Data/main/"
       "Donald%20Trump.txt")
print(requests.get(url, timeout=30).text[:50])

In [ ]:
# 2
guns_url = ("https://raw.githubusercontent.com/fivethirtyeight/guns-data/"
            "master/interactive_data.csv")
guns = pd.read_csv(guns_url, index_col=0)
by_sex = guns[(guns["Intent"] == "None selected")
              & (guns["Age"] == "None selected")
              & (guns["Race"] == "None selected")
              & (guns["Gender"] != "None selected")]
by_sex

In [ ]:
# 3
users = pd.read_csv("user_behavior_dataset.csv")
print(users["Operating System"].value_counts())
users.groupby("Operating System")["App Usage Time (min/day)"].mean()

### Section 2.6: Combining DataFrames

In [ ]:
# 1
users = pd.read_csv("user_behavior_dataset.csv")
android = users[users["Operating System"] == "Android"]
ios = users[users["Operating System"] == "iOS"]
together = pd.concat([android, ios], ignore_index=True)
print(together.shape == users.shape)
print(list(together.columns) == list(users.columns))

In [ ]:
# 2
doctors_csv = pd.read_csv("Egyptian Doctors Data.csv")
renamed = doctors_csv.rename(columns={"fees": "fee"})
mixed = pd.concat([doctors_csv, renamed], ignore_index=True)
print(mixed.shape)

### Section 2.7: Web scraping

In [ ]:
# 1
import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Newsroom training script (you@example.org)"}
response = requests.get("https://quotes.toscrape.com/page/2/",
                        headers=headers, timeout=30)
page_two = BeautifulSoup(response.text, "html.parser")
for quote in parse_quotes(page_two):
    print(quote["author"])

In [ ]:
# 2
response = requests.get("https://quotes.toscrape.com/page/1/",
                        headers=headers, timeout=30)
for quote in parse_quotes(BeautifulSoup(response.text, "html.parser")):
    if "life" in quote["tags"].split(","):
        print(quote["author"], "|", quote["text"][:50])

In [ ]:
# 3
longest = quotes["text"].str.len().idxmax()
print(quotes.loc[longest, "author"])
print(len(quotes.loc[longest, "text"]))

In [ ]:
# 4
print((quotes["tags"].fillna("") == "").sum())

### Section 2.8: RSS feeds

In [ ]:
# 1
enforcement = releases[releases["category"] == "Enforcement Actions"]
print(enforcement[["title", "published"]].head(3))

In [ ]:
# 2
print(releases["published"].dt.day_name().value_counts())

In [ ]:
# 3
from urllib.parse import quote_plus

search = quote_plus("budget when:1d")
print("https://news.google.com/rss/search?q=" + search
      + "&hl=en-GB&gl=GB&ceid=GB:en")

### Section 2.9: Extracting the text of articles

In [ ]:
# 1
import trafilatura

url = "https://quotes.toscrape.com/author/Albert-Einstein/"
html_page = requests.get(url, headers=headers, timeout=30).text
einstein = trafilatura.extract(html_page)
print(len(einstein))

In [ ]:
# 2
as_markdown = trafilatura.extract(html_page, output_format="markdown")
for line in as_markdown.splitlines()[:3]:
    print(line[:70])

In [ ]:
# 3
page_html = requests.get("https://quotes.toscrape.com/page/1/",
                         headers=headers, timeout=30).text
quotes_text = trafilatura.extract(page_html)
for line in quotes_text.splitlines()[:8]:
    print(line[:70])